In [1]:
import pandas as pd
import numpy as np
import re

raw_df = pd.read_excel("../data/raw_data/GSAF5.xls")

print(f"Num rows: {len(raw_df)}")
raw_df.head()

Num rows: 7103


,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,...,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22
0,23rd June,2026.0,Unprovoked,Bahamas,Staniel Cay,Exhuma Cays,Swimming,Unknown,M,12,...,Unknown,Keith Cowley: Kevin McMurray Trackingsharks.co...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,13th June,2026.0,Unprovoked,USA,Florida,Reservation Way near Indian Pass,Swimming,Keira Ralph,F,17,...,Unknown small shark,Keith Cowley: Simon De Marchi: All things Emer...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,13th June,2026.0,Unprovoked,Australia,NSW,Coogee Beach,Swimming,Leah Stewart,F,35,...,Great White Shark 4m,Simon De Marchi: Andrew Currie: ABC NEWS: 9 Ne...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11th June,2026.0,Unprovoked,Galapogos Islands,Santa Fe Island,Mosquera Islet,Snorkeling,Australia Woman,F,?,...,Unknown,Andrew Currie: Facebook: AbsolutCruceros,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,8th June,2026.0,Unprovoked,USA,Florida,Panama City,Swimming,Unknown,M,20's,...,2.4m (8ft) Bull shark,James Kingsley:Todd Smith: Kevin McMurray Trac...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# clean column names
keep_cols = ['Date', 'Type', 'Country', 'State', 'Activity', 'Sex', 'Age', 'Fatal Y/N', 'Time', 'Species ']

raw_df = raw_df[keep_cols]
for col_name in raw_df.columns:
    new_name = col_name.lower().replace("y/n", "").strip()
    raw_df.rename(columns={col_name: new_name}, inplace=True)
raw_df.head()

,date,type,country,state,activity,sex,age,fatal,time,species
0,23rd June,Unprovoked,Bahamas,Staniel Cay,Swimming,M,12,N,1530hrs,Unknown
1,13th June,Unprovoked,USA,Florida,Swimming,F,17,N,1145hrs,Unknown small shark
2,13th June,Unprovoked,Australia,NSW,Swimming,F,35,N,1115hrs,Great White Shark 4m
3,11th June,Unprovoked,Galapogos Islands,Santa Fe Island,Snorkeling,F,?,N,Day,Unknown
4,8th June,Unprovoked,USA,Florida,Swimming,M,20's,N,1145hrs,2.4m (8ft) Bull shark


In [3]:
# filter to unprovoked incidents with "fatality" not unknown
clean_df = raw_df.copy()

clean_df = clean_df[clean_df["type"] == "Unprovoked"]
clean_df = clean_df[clean_df["fatal"].isin(["Y", "N"])]
print(f"Number of unprovoked records: {len(clean_df)}")

Number of unprovoked records: 5164


In [4]:
def clean_by_keyword(attr, keyword_mappings, testing=False):

    raw_attr = str(attr).lower()
    clean_attr = raw_attr if testing else np.nan

    for mapped_word, keyword_list in keyword_mappings.items():
        for keyword in keyword_list:
            if keyword in raw_attr:
                clean_attr = mapped_word

    return clean_attr

In [5]:
clean_activities = clean_df["activity"].copy()

activity_mappings = {
    # "unknown": ["not stated", "unknown", "undisclosed"],
    "swimming": ["swimming", "jumped", "jumping", "swmming", "hawser", "exercise", "playing", "training", "pool ring", 
                    "treading", "floating", "drill", "towing", "snorkeling", "snorkelling"],
    "surfing/bodyboarding": ["surfing", "surfboard", "paddling", "bodyboard", "boogie", "boggie", "surfng", "body board", "body-board"],
    "SUP/foiling": ["sup", "foil", "paddleboarding", "paddle boarding", "paddle-board", "kite board", "kite-board", "kiteboard"],
    "diving": ["dive", "diving"],
    "fishing": ["fishing", "fish", "netting", "hooking", "catching", "fihing", "casting"],
    "dive fishing": ["spear", "lobstering", "scalloping", "crabbing", "lobster"],
    "small watercraft": ["kayak", "raft", "zodiac", "in boat", "surf ski", "surf-ski", "canoe", "kayaying", "kakaying", "jet ski"],
    "adrift": ["fell", "adrift", "wreck", "crash", "capsize", "overturned", "overboard", "washed off", "disaster", "parasail"],
    "wading": ["wading", "washing", "bathing", "ankle-deep", "sitting", "standing", "squatting", "walking", "kneeling", 
                "crouching", "lying", "sittting", "stamding", "ran", "waist-deep", "clamming", "shallow", "defecating"],
    "rescuing": ["rescue", "body recovery"]
}

clean_activities = clean_activities.apply(lambda x: clean_by_keyword(x, activity_mappings))

clean_df["activity"] = clean_activities

In [6]:
species_mappings = {
    # "unknown": ["not stated", "unknown", "undisclosed"],
    "great white": ["great white", "white shark", "white xhark", "wfite shark", "gws"],
    "bull": ["bull", "zambezi", "zambesi", "zambi", "nicaragua", "shovelnose", "bu.ll"],
    "tiger": ["tiger"],
    "blacktip": ["blacktip", "black tip", "black-tip", "carcharhinid"],
    "whitetip": ["whitetip", "white tip"],
    "bronze whaler": ["bronze", "copper", "bronzie", "broze"],
    "nurse": ["nurse"],
    "sevengill": ["sevengill", "seven gill", "seven-gill", "7-gill"],
    "mako": ["mako"],
    "spinner": ["spinner"],
    "lemon": ["lemon"],
    "hammerhead": ["hammerhead"],
    "wobbegong": ["wobbegong", "carpet"],
    "raggedtooth": ["raggedtooth", "ragged-tooth", "ragged tooth", "sand tiger"],
    "dusky": ["dusky"],
    "galapagos": ["galapagos"],
    "blue": ["blue"],
    "reef": ["reef"]
}

clean_species = clean_df["species"].copy()
print(f"Num unique before: {len(clean_species.unique())}")
clean_species = clean_species.apply(lambda x: clean_by_keyword(x, species_mappings, testing=False))
print(f"Num unique after: {len(clean_species.unique())}")

# with open("unique_species.txt", "w") as f:
#     for specie in clean_species.values.unique():
#         f.write(f"{specie}\n")

clean_df["species"] = clean_species

Num unique before: 1282
Num unique after: 19


In [7]:
def clean_date_str(date_str):
    clean_str = date_str.lower().replace("reported", "").replace("ca", "").replace("22nd-", "")
    clean_str = re.sub(r"\s*-+\s*", " ", clean_str)
    clean_str = re.sub(r"[^\w\s]", "", clean_str)
    return clean_str


clean_dates = clean_df["date"].copy().astype("string")

# get mask for values that only list a year and no month
yearonly_mask_list = []
for raw_date in clean_dates:
    yearonly_mask_list.append(bool(re.fullmatch(r"\d{4}", raw_date)))

clean_dates = clean_dates.apply(lambda x: clean_date_str(x))
clean_dates = pd.to_datetime(clean_dates, errors="coerce", format="mixed")
clean_df["clean_date"] = clean_dates


clean_df["year"] = clean_df["clean_date"].dt.year.astype("Int64")
clean_df["year"] = clean_df["year"].replace(1, np.nan)
clean_df["month"] = clean_df["clean_date"].dt.month.astype("Int64")

clean_df.loc[yearonly_mask_list, "month"] = np.nan

clean_df.drop(["date", "clean_date"], axis=1, inplace=True)

In [8]:
time_mappings = {
    "evening": ["evening", "dusk", "sunset", "pm", "p.m.", "before sundown"],
    "afternoon": ["afternoon", "noon", "midday", "day", "lunctime", "afternon"],
    "morning": ["morning", "am", "a.m.", "dawn", "daybreak"],
    "night": ["night", "nigt", "after dark"]
}

def clean_time(raw_time):
    time_str = str(raw_time).lower().strip()
    time_str = time_str.replace("hrs", "").replace("h", "").replace("j", "").replace("11oo", "1100")

    digits_search = re.search(r"\d{3,}", time_str)

    if digits_search:
        time_num = int(digits_search.group())
        if 500 <= time_num < 1200:
            return "morning"
        elif 1200 <= time_num < 1700:
            return "afternoon"
        elif 1700 <= time_num < 2100:
            return "evening"
        else:
            return "night"
        
    return clean_by_keyword(time_str, time_mappings, testing=False)


clean_times = clean_df["time"].copy()
clean_times = clean_times.apply(lambda x: clean_time(x))

# with open("unique_times.txt", "w") as f:
#     for time in clean_times.unique():
#         f.write(f"{time}\n")

clean_df["time"] = clean_times

In [9]:
clean_sex = clean_df["sex"].copy()

def clean_sex_val(raw_val):
    clean_val = str(raw_val).lower()
    return clean_val if clean_val in ["m", "f"] else np.nan

clean_sex = clean_sex.apply(lambda x: clean_sex_val(x))
clean_df["sex"] = clean_sex

In [10]:
clean_age = clean_df["age"].copy()

def clean_age_val(raw_val):

    raw_str = str(raw_val)

    if bool(re.fullmatch(r"\d+", raw_str)):
        return int(raw_str)

    if bool(re.fullmatch(r"\d+s", raw_str)):
        return int(raw_str[0] + "5")

    return np.nan

clean_age = clean_age.apply(lambda x: clean_age_val(x))
clean_df["age"] = clean_age

In [11]:
clean_state = clean_df["state"].copy()

def clean_state_val(raw_val):

    new_str = str(raw_val).lower().strip()

    return new_str

clean_state = clean_state.apply(lambda x: clean_state_val(x))

with open("unique_states.txt", "w") as f:
    for state in clean_state.unique():
        f.write(f"{state}\n")

In [12]:
clean_country = clean_df["country"].copy()

def clean_country_val(raw_val):

    new_str = str(raw_val).lower().replace("?", "").strip()

    if "/" in new_str:
        new_str = new_str.split(" / ")[0]

    if new_str in ["england", "scotland", "united kingdom"]:
        new_str = "united kingdom"

    return new_str

clean_country = clean_country.apply(lambda x: clean_country_val(x))

with open("unique_countries.txt", "w") as f:
    for country in clean_country.unique():
        f.write(f"{country}\n")

In [18]:
clean_df.to_csv("../data/clean_data/clean_data.csv", index=False)